# LatentMind V6 — Colab Notebook

**Training-first pipeline**: synthesizes agentic traces → trains the policy brain → loads the agent → runs tests.

The brain is a trained 3-head MLP. It picks one action at a time — rag · sql · chart · email · template — and re-decides after each step, stopping when its *continue* score drops below the seuil.

Requires: T4 or L4 GPU runtime.

In [23]:
# ─── GPU CLEANUP — run this any time before re-running the agent cells ───────
# Releases all model weights from VRAM so you can reload without OOM.
# Safe to run even on the very first pass (does nothing if nothing is loaded).

import gc, sys, torch

def cleanup(verbose: bool = True) -> None:
    freed: list[str] = []

    # SLM (3B main model + polisher)
    try:
        import v6.slm as _m
        if _m._slm is not None:
            for tid in list(_m._slm._store.keys()):
                _m._slm.clear_thread(tid)
            del _m._slm.model
            _m._slm = None
            freed.append("SLM 3B")
        if _m._polisher is not None:
            del _m._polisher.model
            _m._polisher = None
            freed.append("polisher 0.5B")
    except Exception:
        pass

    # BGE-M3 encoder + retriever
    try:
        import v6.knowledge as _m
        if _m._encoder is not None:
            del _m._encoder.model
            _m._encoder = None
            freed.append("BGE-M3")
        _m._retriever = None
    except Exception:
        pass

    # Brain + schema singletons
    try:
        import v6.brain as _m
        if _m._brain is not None:
            _m._brain = None
            freed.append("brain head")
    except Exception:
        pass
    try:
        import v6.schema as _m
        _m._schema = None
    except Exception:
        pass

    # Garbage-collect + CUDA cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        alloc  = torch.cuda.memory_allocated() / 1e9
        total  = torch.cuda.get_device_properties(0).total_memory / 1e9
        status = f"{alloc:.2f} GB / {total:.1f} GB in use after cleanup"
    else:
        status = "no CUDA device"

    if verbose:
        if freed:
            print("Released:", ", ".join(freed))
        else:
            print("Nothing was loaded — nothing to release")
        print(status)
        print("Ready to re-run Cell 7.")

cleanup()

Released: SLM 3B, polisher 0.5B, BGE-M3, brain head
6.39 GB / 15.6 GB in use after cleanup
Ready to re-run Cell 7.


In [24]:
# Core inference + RAG
!pip install 'transformers>=4.46.0' 'sentence-transformers>=3.0.0' 'accelerate>=0.27.0'
print("✓ transformers, sentence-transformers, accelerate")

# Graph + math
!pip install 'langgraph>=0.2.0' 'bitsandbytes>=0.43.0' scipy matplotlib
print("✓ langgraph, bitsandbytes, scipy, matplotlib")

# Utils
!pip install jinja2 pydantic pymysql mysql-connector-python
print("✓ jinja2, pydantic, mysql connectors")

print("\n✓ Setup complete! Ready to load agent.")

✓ transformers, sentence-transformers, accelerate
✓ langgraph, bitsandbytes, scipy, matplotlib
✓ jinja2, pydantic, mysql connectors

✓ Setup complete! Ready to load agent.


In [25]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, shutil

# ── GitHub token (required for private repo) ─────────────────────────
# Add a Colab secret named GITHUB_TOKEN (the key icon in the left panel)
# with a Fine-Grained PAT that has read access to Latent-Djezzy.
# Alternatively, make the repository public and remove the token logic.
try:
    from google.colab import userdata
    _tok = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    _tok = ''

if not _tok:
    import getpass
    print('GITHUB_TOKEN secret not found — enter a personal access token')
    print('(leave blank only if the repo is public)')
    _tok = getpass.getpass('GitHub token: ')

REPO_BASE = 'https://github.com/Hamza09Hamza/Latent-Djezzy.git'
REPO_URL  = f'https://{_tok}@github.com/Hamza09Hamza/Latent-Djezzy.git' if _tok else REPO_BASE
REPO_DIR  = '/content/Latent-Djezzy'
BRANCH    = 'v6-clean'

# Always ensure we have the correct branch — delete and re-clone on first run
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('cleaned up old repo')

# Clone — token is embedded in the URL so no interactive prompt
subprocess.run(
    ['git', 'clone', '--depth=1', '--branch', BRANCH, REPO_URL, REPO_DIR],
    check=True, capture_output=True)
print(f'cloned {BRANCH} → {REPO_DIR}')

# Clear cached modules so reimports get fresh code
for mod in list(sys.modules.keys()):
    if 'v6' in mod:
        del sys.modules[mod]

os.chdir(REPO_DIR)
print('repo ready:', REPO_DIR)

# Check if the trained brain head exists
head_path = 'models/brain_head.pt'
if os.path.isfile(head_path):
    size_mb = os.path.getsize(head_path) / 1e6
    print(f'✓ Trained brain head found ({size_mb:.1f} MB) — will skip training')
else:
    print('! Trained brain head NOT found — Cells 6–7 will build it (~2 min on T4)')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cleaned up old repo


CalledProcessError: Command '['git', 'clone', '--depth=1', '--branch', 'v6-clean', 'https://github.com/Hamza09Hamza/Latent-Djezzy.git', '/content/Latent-Djezzy']' returned non-zero exit status 128.

In [ ]:
import shutil, os

# Database location — checks LatentDjezzy folder first, falls back to MyDrive root
LOCAL_DB   = "/content/interndb.sqlite"

possible_locations = [
    "/content/drive/MyDrive/LatentDjezzy/interndb.sqlite",
    "/content/drive/MyDrive/interndb.sqlite",
]

DRIVE_DB = None
for loc in possible_locations:
    if os.path.isfile(loc):
        DRIVE_DB = loc
        break

if not DRIVE_DB:
    print("⚠ Database not found in:")
    for loc in possible_locations:
        print(f"    {loc}")
    print("\nPlease ensure interndb.sqlite is in /MyDrive/LatentDjezzy/ or /MyDrive/")
else:
    if not os.path.isfile(LOCAL_DB):
        shutil.copy(DRIVE_DB, LOCAL_DB)
        print(f"✓ copied {os.path.getsize(LOCAL_DB):,} bytes → {LOCAL_DB}")
    else:
        print("✓ SQLite already present:", LOCAL_DB)

# Create output dirs on Drive so charts and emails are persisted
output_base = "/content/drive/MyDrive/LatentDjezzy/v6_output"
for d in [
    f"{output_base}/charts",
    f"{output_base}/emails",
    f"{output_base}/reports",
]:
    os.makedirs(d, exist_ok=True)
print(f"✓ Output dirs ready on Drive → {output_base}/")

In [ ]:
import os, warnings, logging

# Suppress noisy transformers deprecation warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

os.environ["V6_USE_SQLITE"]    = "1"
os.environ["V6_SQLITE_PATH"]   = "/content/interndb.sqlite"
os.environ["V6_4BIT"]          = "0"          # f16 — faster on T4, history cap prevents OOM
os.environ["V6_SPECULATIVE"]   = "0"          # off — drafter wastes VRAM alongside 3B
os.environ["V6_SLM_OVERRIDE"]  = "Qwen/Qwen2.5-Coder-3B-Instruct"
os.environ["V6_OUTPUT_DIR"]    = "/content/drive/MyDrive/LatentDjezzy/v6_output"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {device}  ({total_gb:.1f} GB)")
print(f"Model:              3B f16 (~6.4 GB) — history capped to 2 turns to prevent OOM")
print(f"Speculative dec.:   OFF")
print(f"Output dir:         /content/drive/MyDrive/LatentDjezzy/v6_output")

In [ ]:
import os

head_path = "models/brain_head.pt"
if os.path.isfile(head_path):
    print("Skipping trace synthesis (trained brain head already exists)")
else:
    # Synthesize agentic traces — the editable policy spec — for the brain.
    # Output: v6/data/brain_train.jsonl
    !python3 -m v6.brain_data
    !wc -l v6/data/brain_train.jsonl

In [ ]:
import os

head_path = "models/brain_head.pt"
if os.path.isfile(head_path):
    print("Brain head already trained; skipping training")
else:
    # Train the 3-head MLP (intent · action · continue) — ~160 epochs,
    # a couple of minutes on T4. Output: models/brain_head.pt
    !python3 -m v6.train_brain

print("Brain ready." if os.path.isfile(head_path)
      else "! Training did not produce models/brain_head.pt — check the log above")

In [ ]:
import sys, os, torch
sys.path.insert(0, "/content/Latent-Djezzy")

# Free any previously loaded models before (re-)loading — prevents OOM on re-run
try:
    cleanup(verbose=False)
except NameError:
    pass  # cleanup not defined yet (first run) — that's fine

from v6.graph import LatentMindV6
from v6.slm import get_slm, get_polisher
from v6.brain import get_brain

print("Loading BGE-M3 encoder + SLM (downloads on first run)…")
agent = LatentMindV6()
get_slm()    # force-load the main model so VRAM is allocated before queries
get_brain()  # force-load the trained brain head (fails fast if not trained)

print("\nLoading polisher (0.5B — ~1 GB, streams the final answer)…")
try:
    get_polisher()
    print("✓ Polisher ready")
except Exception as _e:
    print(f"  Polisher unavailable ({_e}) — raw answers will be shown instead")

if torch.cuda.is_available():
    alloc_gb = torch.cuda.memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU: {alloc_gb:.1f} GB / {total_gb:.1f} GB  "
          f"(headroom ~{total_gb - alloc_gb:.1f} GB)")

In [ ]:
from v6.config import V6Config

print("\n" + "="*60)
print("Configuration Verification")
print("="*60)
print(f"SLM model:            {V6Config.slm_id()}")
print(f"4-bit quantization:   {'✓ Enabled' if V6Config.USE_4BIT else '✗ Disabled'}")
print(f"Brain head:           {V6Config.BRAIN_HEAD_PATH}")
print(f"Brain seuil:          {V6Config.BRAIN_SEUIL}  (continue ≥ this → keep going)")
print(f"Brain max steps:      {V6Config.BRAIN_MAX_STEPS}")
print("="*60 + "\n")
print("Ready to run queries. Try: ask('Hello, what can you do?')")

In [ ]:
import time, os, sys
from v6.state import initial_state
from IPython.display import display, Image, Markdown

_DIM, _RESET = "\033[2m", "\033[0m"

def _type(text, delay=0.005):
    # typewriter effect for the dim "thinking" lines
    for ch in text:
        sys.stdout.write(ch)
        sys.stdout.flush()
        time.sleep(delay)

def ask(question, thread="main"):
    # One turn: stream the brain's thinking, then the polished answer.
    config = {"configurable": {"thread_id": thread}, "recursion_limit": 60}
    state  = initial_state(question, thread)

    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    t0 = time.time()

    shown = 0
    final_answer  = ""
    chart_path    = ""
    document_path = ""
    email_draft   = None
    intent        = ""
    timings       = {}

    for event in agent.graph.stream(state, config=config, stream_mode="updates"):
        for node, data in event.items():
            if not data:
                continue
            # stream any new thoughts as dim "thinking" lines
            thoughts = data.get("thoughts")
            if thoughts and len(thoughts) > shown:
                for th in thoughts[shown:]:
                    if th.get("kind") == "thinking":
                        sys.stdout.write(_DIM + "  💭 ")
                        _type(th["text"])
                        print(_RESET)
                shown = len(thoughts)
            if data.get("timings"):
                timings = data["timings"]
            # transparent view of the brain's decision each tick
            if node == "brain":
                intent = data.get("intent", intent)
                act    = data.get("next_action", "")
                cont   = data.get("continue_score", 0.0)
                print(f"{_DIM}     → {act}  (continue {cont:.2f}){_RESET}")
            elif node == "chart":
                chart_path = data.get("chart_path", "") or chart_path
            elif node == "template":
                document_path = data.get("document_path", "") or document_path
            elif node == "email":
                email_draft = data.get("email_draft") or email_draft
            elif node == "communicator":
                final_answer = data.get("final_answer", "")

    elapsed = time.time() - t0

    # ── stream the polished answer as the normal message ──────────────────
    print(f"\nAnswer ({elapsed:.1f}s):")
    _FAILURE = ("couldn't build", "failed to run", "no matching rows",
                "wasn't able to pull", "not sure which kpi")
    is_failure = any(p in (final_answer or "").lower() for p in _FAILURE)
    should_polish = (intent == "data" and final_answer
                     and len(final_answer) > 40 and not is_failure
                     and not chart_path and not email_draft
                     and not document_path)
    if should_polish:
        try:
            from v6.slm import get_polisher
            for token in get_polisher().stream(final_answer, question):
                print(token, end="", flush=True)
            print()
        except Exception:
            print(final_answer)
    else:
        print(final_answer or "(no answer)")

    # ── display chart inline ──────────────────────────────────────────────
    if chart_path and os.path.isfile(chart_path):
        print()
        display(Image(chart_path))
    elif chart_path:
        print(f"  (chart path reported but file missing: {chart_path})")

    # ── report ────────────────────────────────────────────────────────────
    if document_path:
        print(f"  Report saved → {document_path}")

    # ── email draft + save to Drive ───────────────────────────────────────
    if email_draft:
        print()
        to = email_draft.get("to") or "(no recipient matched)"
        display(Markdown(f"**Email draft**\n\n"
                         f"**To:** {email_draft.get('to_name', '?')} <{to}>  \n"
                         f"**Subject:** {email_draft.get('subject', '')}\n\n"
                         f"---\n\n{email_draft.get('body', '')}"))
        try:
            import datetime
            from v6.config import V6Config
            email_dir = os.path.join(V6Config.output_dir(), "emails")
            os.makedirs(email_dir, exist_ok=True)
            stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            email_path = os.path.join(email_dir, f"email_{stamp}.txt")
            with open(email_path, "w") as f:
                f.write(f"To: {email_draft.get('to_name', '?')} <{to}>\n")
                f.write(f"Subject: {email_draft.get('subject', '')}\n\n")
                f.write(email_draft.get("body", ""))
            print(f"  Email saved → {email_path}")
        except Exception as e:
            print(f"  (could not save email: {e})")

    # ── performance breakdown — where the T4 time actually went ───────────
    import torch
    brain_ms = sum(v for k, v in timings.items() if k.startswith("brain"))
    sql_ms   = sum(v for k, v in timings.items() if k.startswith("sql"))
    rag_ms   = timings.get("rag_ms", 0.0)
    n_ticks  = sum(1 for k in timings if k.startswith("brain"))
    line = (f"⏱  {n_ticks} brain ticks {brain_ms:.0f}ms · "
            f"rag {rag_ms / 1000:.2f}s · sql {sql_ms / 1000:.2f}s · "
            f"total {elapsed:.1f}s")
    if torch.cuda.is_available():
        line += f" · VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB"
    print(f"{_DIM}{line}{_RESET}")

    # ── free the KV cache to keep VRAM flat across queries ────────────────
    try:
        from v6.slm import get_slm
        get_slm().clear_thread(thread)
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print()

In [ ]:
ask("Hello, what can you do?")

In [ ]:
ask("What is ARPU?")

In [ ]:
ask("What is the total revenue for Oran in 2025?")

In [ ]:
ask("Show me a bar chart of total revenue by wilaya for 2025")

In [ ]:
ask("Compare churn rates between Algiers and Constantine last quarter")

In [ ]:
ask("Generate an executive report for Q4 2025 performance")

In [ ]:
ask("Email the total revenue for Oran to the finance director")

In [ ]:
ask("and for Constantine?")  # follow-up — tests memory

In [ ]:
ask("What is the satellite coverage ratio for Oran?")

In [ ]:
ask("Chart the ebitda by wilaya for 2025 and email it to the finance director")

In [ ]:
# Times agent.graph.invoke directly (no UI polish / typewriter) so the numbers
# are the real engine cost. The brain MLP is milliseconds — the SLM dominates.
import time, torch
from v6.state import initial_state
from v6.slm import get_slm

BENCH = [
    "Hello, what can you do?",
    "What is ARPU?",
    "What is the total revenue for Oran in 2025?",
    "Show me a bar chart of total revenue by wilaya for 2025",
    "Compare churn rates between Algiers and Constantine",
    "Email the total revenue for Oran to the finance director",
]

hdr = f"{'query':<46}{'intent':<11}{'steps':>6}{'brain':>9}{'sql':>9}{'total':>9}"
print(hdr)
print("-" * len(hdr))
tot = 0.0
for q in BENCH:
    t0 = time.time()
    r = agent.graph.invoke(
        initial_state(q, "bench"),
        {"configurable": {"thread_id": "bench"}, "recursion_limit": 60})
    dt = time.time() - t0
    tot += dt
    tm = r.get("timings", {})
    brain_ms = sum(v for k, v in tm.items() if k.startswith("brain"))
    sql_ms   = sum(v for k, v in tm.items() if k.startswith("sql"))
    print(f"{q[:46]:<46}{r.get('intent',''):<11}"
          f"{r.get('brain_step',0):>6}{brain_ms:>7.0f}ms"
          f"{sql_ms/1000:>8.2f}s{dt:>8.2f}s")
    get_slm().clear_thread("bench")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("-" * len(hdr))
print(f"{'AVERAGE per query':<46}{'':<11}{'':>6}{'':>9}{'':>9}"
      f"{tot/len(BENCH):>8.2f}s")
if torch.cuda.is_available():
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM: {a:.2f} GB used / {t:.1f} GB total  ({t-a:.1f} GB free)")
    print("Brain overhead is the 'brain' column — a few ms per query; the "
          "SLM router+sqlgen is the cost centre.")

In [ ]:
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated()  / 1e9
    reserv = torch.cuda.memory_reserved()   / 1e92
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  allocated: {alloc:.2f} GB")
    print(f"GPU  reserved:  {reserv:.2f} GB")
    print(f"GPU  total:     {total:.2f} GB")

In [ ]:
# Direct streaming — bypasses the graph, useful for quick SLM checks.
from v6.slm import get_slm

slm = get_slm()
messages = [
    {"role": "system", "content": "You are a helpful telecom analyst."},
    {"role": "user",   "content": "List the top 3 KPIs for a telecom operator."},
]
print("Streaming response:")
for token in slm.stream_generate(messages, max_new_tokens=256):
    print(token, end="", flush=True)
print()